# mBERT — Hindi Fake News Detection (paper-faithful replication)

replicating the baseline from the CONSTRAINT 2021 paper using `bert-base-multilingual-cased`.  
trying to hit or beat their reported MCC 0.664 / F1 0.728.

**key choices (matching paper exactly):**
- batch size 32, 5 epochs, lr 2e-5, dropout 0.1
- 2-layer MLP head (2 hidden units each, ReLU) replacing default softmax head
- weighted CE for class imbalance (~20% fake)
- linear warmup schedule
- checkpoint saved on val MCC — more informative than val loss for imbalanced data

**stuff i removed from earlier versions:**
- data augmentation (made things worse on hindi — too noisy)
- threshold calibration (was leaking val info, killed test score)
- LLRD (layer-wise LR decay — added instability, no clear gain)


In [ ]:
import os, re, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
)

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# matching paper config exactly — don't touch EPOCHS or BATCH_SIZE
MODEL_NAME  = "bert-base-multilingual-cased"
MAX_LEN     = 256
BATCH_SIZE  = 32       # paper uses 32
EPOCHS      = 5        # paper uses 5
LR          = 2e-5     # paper uses 2e-5
WARMUP_FRAC = 0.1
DROPOUT     = 0.1      # paper default
SEED        = 42

TRAIN_PATH = "/kaggle/input/datasets/sherwinmazarello/hindimb/cleaned_file2.csv"
VALID_PATH = "/kaggle/input/datasets/sherwinmazarello/hindimb/cleaned_dataset_valid.csv"
TEST_PATH  = "/kaggle/input/datasets/sherwinmazarello/hindimb/cleaned_dataset_test.csv"

CKPT_PT   = "/kaggle/working/best_mbert_v3.pt"
CKPT_HF   = "/kaggle/working/best_mbert_v3_hf"
ZIP_PATH  = "/kaggle/working/mbert_v3_download.zip"
PROBS_DIR = "/kaggle/working/ensemble_probs"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", device)


In [ ]:
def convert_label(x):
    return 1 if "fake" in str(x).lower() else 0

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\sऀ-ॿ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def prepare_df(path):
    df = pd.read_csv(path).copy()
    df["label"] = df["Labels Set"].apply(convert_label)
    df["text"]  = df["Post"].astype(str).apply(clean_text)
    df = df.drop_duplicates(subset="text")
    df = df[df["text"].str.split().str.len() > 5]
    return df[["text", "label"]].reset_index(drop=True)

train_df = prepare_df(TRAIN_PATH)
valid_df = prepare_df(VALID_PATH)
test_df  = prepare_df(TEST_PATH)

print("dataset breakdown:")
for name, df in [("Train", train_df), ("Valid", valid_df), ("Test", test_df)]:
    total = len(df)
    fake  = int(df["label"].sum())
    real  = total - fake
    print(f"  {name}: total={total}  fake={fake} ({fake/total*100:.1f}%)  real={real} ({real/total*100:.1f}%)")

n_fake    = int(train_df["label"].sum())
n_nonfake = len(train_df) - n_fake
weight_fake    = len(train_df) / (2 * n_fake)
weight_nonfake = len(train_df) / (2 * n_nonfake)
class_weights  = torch.tensor([weight_nonfake, weight_fake], dtype=torch.float).to(device)
print(f"\nclass weights: real={weight_nonfake:.3f}  fake={weight_fake:.3f}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HindiNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts   = df["text"].tolist()
        self.labels  = df["label"].tolist()
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = HindiNewsDataset(train_df, tokenizer, MAX_LEN)
valid_ds = HindiNewsDataset(valid_df, tokenizer, MAX_LEN)
test_ds  = HindiNewsDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"loaders ready — train={len(train_loader)} batches, val={len(valid_loader)}, test={len(test_loader)}")


## Model: mBERT + 2×2 MLP head

Paper (section 3.2.1) says they replaced the default softmax head with a 2-layer MLP,  
2 hidden units per layer, ReLU activations.  

So the full head is: `[CLS] → Dropout → Linear(768→2) → ReLU → Linear(2→2) → ReLU → Linear(2→2)`  

The final linear goes to 2 logits (binary), loss is weighted CE (no softmax inside model, applied in loss).


In [ ]:
class MBertWithMLPHead(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        hidden_size     = self.bert.config.hidden_size  # 768 for mBERT

        # paper: 2 hidden layers, 2 neurons each, ReLU
        self.mlp = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
            nn.ReLU(),
            nn.Linear(2, 2),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(2, 2)

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # [CLS] token
        return self.classifier(self.mlp(cls_output))

    def get_probs(self, input_ids, attention_mask):
        return F.softmax(self.forward(input_ids, attention_mask), dim=-1)


model = MBertWithMLPHead(MODEL_NAME, dropout=DROPOUT).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params:     {total:,}")
print(f"trainable params: {trainable:,}")


In [ ]:
# AdamW with uniform lr (no LLRD — tried it, didn't help here)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"total steps: {total_steps}  warmup: {warmup_steps}")


In [ ]:
def evaluate(model, loader, criterion, device, return_probs=False):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits     = model(input_ids, attention_mask)
            loss       = criterion(logits, labels)
            total_loss += loss.item()

            probs = F.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=1)

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    metrics = {
        "loss":      total_loss / len(loader),
        "accuracy":  accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, pos_label=1, zero_division=0),
        "recall":    recall_score(all_labels, all_preds, pos_label=1, zero_division=0),
        "f1":        f1_score(all_labels, all_preds, pos_label=1, zero_division=0),
        "macro_f1":  f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "mcc":       matthews_corrcoef(all_labels, all_preds),
    }

    if return_probs:
        return metrics, all_probs, all_labels
    return metrics


In [ ]:
# training loop — saving best checkpoint on val MCC
# MCC is more meaningful than F1 on imbalanced data

history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [], "val_mcc": []}
best_val_mcc = -1.0
best_epoch   = -1

print(f"training: {EPOCHS} epochs  batch={BATCH_SIZE}  lr={LR}")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_preds, train_labels = [], []

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss_sum += loss.item()
        train_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss = train_loss_sum / len(train_loader)
    train_f1   = f1_score(train_labels, train_preds, average="macro", zero_division=0)
    val_m      = evaluate(model, valid_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_m["loss"])
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_m["macro_f1"])
    history["val_mcc"].append(val_m["mcc"])

    print(f"epoch {epoch}/{EPOCHS}  "
          f"train_loss={train_loss:.4f}  val_loss={val_m['loss']:.4f}  "
          f"val_f1={val_m['macro_f1']:.4f}  val_mcc={val_m['mcc']:.4f}")

    if val_m["mcc"] > best_val_mcc:
        best_val_mcc = val_m["mcc"]
        best_epoch   = epoch
        torch.save(model.state_dict(), CKPT_PT)
        os.makedirs(CKPT_HF, exist_ok=True)
        model.bert.save_pretrained(CKPT_HF)
        tokenizer.save_pretrained(CKPT_HF)
        print(f"  -> saved best (MCC={best_val_mcc:.4f})")

print(f"\nbest epoch: {best_epoch}  val MCC: {best_val_mcc:.4f}")


In [ ]:
# reload best weights
model.load_state_dict(torch.load(CKPT_PT, map_location=device))

for split_name, loader in [("Train", train_loader), ("Validation", valid_loader), ("Test", test_loader)]:
    m, probs, labels = evaluate(model, loader, criterion, device, return_probs=True)
    preds = probs.argmax(axis=1)
    print(f"\n{split_name}")
    print(f"  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  macro_f1={m['macro_f1']:.4f}  mcc={m['mcc']:.4f}")


In [ ]:
# training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["val_loss"],   marker="s", label="val loss")
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss")
axes[0].legend()

axes[1].plot(history["val_f1"],  marker="o", color="green", label="val macro-F1")
axes[1].plot(history["val_mcc"], marker="s", color="orange", label="val MCC")
axes[1].set(xlabel="epoch", title="Val Metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# zip model for download
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(CKPT_HF):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
print(f"zip saved: {ZIP_PATH}")
